<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum06/BD_A_P06_2411531001_Habibi%20PutraRizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum06/BD_A_P06_2411531001_Habibi%20Putra%20Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM BIG DATA 6
## Apache Spark: RDD, DataFrame, Spark SQL, dan Lazy Evaluation

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** A

## D. Persiapan Praktikum

Praktikum memakai Google Colab, jadi Apache Spark tidak perlu dipasang manual. PySpark diinstal, lalu SparkSession dibuat dan konfigurasinya dicek.

**Langkah 3.** Instal PySpark.

In [ ]:
!pip install pyspark

**Langkah 4.** Import library dan inisialisasi SparkSession.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PraktikumBigData") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)

Versi Spark: 4.0.5


**Langkah 5.** Periksa konfigurasi Spark.

In [ ]:
print("Spark Application Name:",
      spark.sparkContext.appName)

print("Spark Master:",
      spark.sparkContext.master)

Spark Application Name: PraktikumBigData
Spark Master: local[*]


### Pembahasan D

Hasil inisialisasi menunjukkan bahwa SparkSession berhasil dibuat menggunakan Spark versi 4.0.5. Nama aplikasi yang tercetak adalah `PraktikumBigData`, dengan konfigurasi master `local[*]`. Konfigurasi ini menjalankan Spark secara lokal pada mesin Colab menggunakan seluruh core CPU yang tersedia. Dengan demikian, percobaan dilakukan pada satu mesin, meskipun Spark mendukung pemrosesan pada cluster.

## E.1 Mengenal RDD pada Apache Spark

RDD dibuat dari list angka, lalu dicoba operasi transformation (`map()`, `filter()`) dan action (`count()`, `collect()`).

**Langkah 1.** Membuat RDD.

In [ ]:
data = [10, 20, 30, 40, 50,
        60, 70, 80, 90, 100]

rdd = spark.sparkContext.parallelize(data)

print("Jumlah elemen:", rdd.count())

Jumlah elemen: 10


**Langkah 2.** Melihat isi RDD.

In [ ]:
print(rdd.collect())

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


**Langkah 3.** Transformation `map()`, setiap elemen dikalikan 2.

In [ ]:
rdd_hasil = rdd.map(lambda x: x * 2)

print(rdd_hasil.collect())

[20, 40, 60, 80, 100, 120, 140, 160, 180, 200]


**Langkah 4.** Transformation `filter()`, ambil elemen yang lebih besar dari 50.

In [ ]:
rdd_filter = rdd.filter(lambda x: x > 50)

print(rdd_filter.collect())

[60, 70, 80, 90, 100]


**Langkah 5.** Menggabungkan beberapa transformation, ambil bilangan genap lalu kalikan 3.

In [ ]:
rdd_genap = rdd.filter(lambda x: x % 2 == 0)

rdd_kali = rdd_genap.map(lambda x: x * 3)

print(rdd_kali.collect())

[30, 60, 90, 120, 150, 180, 210, 240, 270, 300]


### Pembahasan E.1

RDD yang dibuat melalui `parallelize()` berisi 10 angka, yaitu 10 sampai 100 dengan selisih 10. Operasi `map()` mengalikan setiap elemen dengan 2 sehingga menghasilkan 20, 40, hingga 200. Sementara itu, `filter()` dengan kondisi `x > 50` menghasilkan lima elemen, yaitu 60, 70, 80, 90, dan 100.

Pada penggabungan operasi, seluruh elemen lolos dari filter bilangan genap karena data awal memang berisi angka genap. Elemen tersebut kemudian dikalikan 3, sehingga diperoleh 30, 60, hingga 300 dengan selisih 30.

#### Pertanyaan langkah 2

`collect()` mengambil seluruh elemen RDD dari semua partisi dan mengembalikannya ke driver dalam bentuk list Python. Operasi ini termasuk action karena memicu eksekusi rangkaian transformation yang diperlukan untuk memperoleh hasil.

#### Pertanyaan langkah 5

1. Transformation, seperti `map()`, `filter()`, dan `flatMap()`, membentuk RDD baru tanpa langsung menghitung hasilnya. Action, seperti `collect()`, `count()`, `take()`, dan `reduce()`, memicu pemrosesan untuk menghasilkan nilai.
2. Operasi `map()` dan `filter()` ditunda karena Spark menerapkan lazy evaluation. Saat keduanya didefinisikan, Spark mencatat hubungan operasi pada RDD. Pemrosesan datanya dimulai ketika action dipanggil.
3. Jika `collect()` pada `rdd_kali` dihapus dan tidak ada action lain yang menggunakan RDD tersebut, rangkaian filter dan perkalian itu belum dieksekusi. Namun, action `count()` dan `collect()` pada sel sebelumnya tetap memproses RDD yang dipanggil masing-masing.
4. Penggunaan `collect()` pada data besar berisiko menghabiskan memori driver karena seluruh hasil dikumpulkan pada satu proses. Pada cluster, pemindahan hasil juga menambah transfer data melalui jaringan. Untuk memeriksa sebagian data dapat digunakan `take(n)`, sedangkan `count()` digunakan jika hanya memerlukan jumlah elemen. Hasil lengkap yang besar sebaiknya disimpan melalui operasi penulisan terdistribusi.

## E.2 Membuat dan Mengolah DataFrame

Dataset transaksi penjualan berisi 10 baris dibuat dari list Python, lalu diolah dengan operasi dasar DataFrame.

**Langkah 1.** Membuat dataset.

In [ ]:
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = [
    "id_transaksi",
    "produk",
    "kategori",
    "harga",
    "jumlah"
]

df = spark.createDataFrame(data_transaksi, kolom)

**Langkah 2.** Menampilkan data.

In [ ]:
df.show()

+------------+---------+----------+-------+------+
|id_transaksi|   produk|  kategori|  harga|jumlah|
+------------+---------+----------+-------+------+
|           1|   Laptop|Elektronik|7500000|     2|
|           2|    Mouse| Aksesoris| 150000|     5|
|           3| Keyboard| Aksesoris| 300000|     3|
|           4|  Monitor|Elektronik|2500000|     2|
|           5|Flashdisk| Aksesoris|  80000|    10|
|           6|  Printer|Elektronik|1800000|     1|
|           7|  Headset| Aksesoris| 250000|     4|
|           8|   Laptop|Elektronik|7500000|     1|
|           9|    Mouse| Aksesoris| 150000|     8|
|          10|  Monitor|Elektronik|2500000|     3|
+------------+---------+----------+-------+------+



**Langkah 3.** Melihat struktur DataFrame.

In [ ]:
df.printSchema()

root
 |-- id_transaksi: long (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- jumlah: long (nullable = true)



**Langkah 4.** Menghitung jumlah baris.

In [ ]:
print("Jumlah transaksi:", df.count())

Jumlah transaksi: 10


**Langkah 5.** Memilih kolom produk, kategori, dan harga.

In [ ]:
df.select(
    "produk",
    "kategori",
    "harga"
).show()

+---------+----------+-------+
|   produk|  kategori|  harga|
+---------+----------+-------+
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
| Keyboard| Aksesoris| 300000|
|  Monitor|Elektronik|2500000|
|Flashdisk| Aksesoris|  80000|
|  Printer|Elektronik|1800000|
|  Headset| Aksesoris| 250000|
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
|  Monitor|Elektronik|2500000|
+---------+----------+-------+



**Langkah 6.** Filtering, transaksi dengan harga produk lebih dari Rp1.000.000.

In [ ]:
df.filter(
    df.harga > 1000000
).show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



**Langkah 7.** Membuat kolom baru, Total = Harga x Jumlah.

In [ ]:
from pyspark.sql.functions import col

df_total = df.withColumn(
    "total",
    col("harga") * col("jumlah")
)

df_total.show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



**Langkah 8.** Mengurutkan transaksi berdasarkan total dari terbesar ke terkecil.

In [ ]:
df_total.orderBy(
    col("total").desc()
).show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
+------------+---------+----------+-------+------+--------+



### Pembahasan E.2

DataFrame yang dibuat berisi 10 transaksi dengan lima kolom. Hasil `printSchema()` menunjukkan bahwa `id_transaksi`, `harga`, dan `jumlah` bertipe `long`, sedangkan `produk` dan `kategori` bertipe `string`. Spark menentukan tipe tersebut secara otomatis dari nilai pada list Python. Jumlah baris juga sesuai dengan hasil `count()`, yaitu 10 transaksi.

Operasi `select()` menampilkan kolom produk, kategori, dan harga tanpa mengubah jumlah baris. Selanjutnya, filter `harga > 1000000` menyisakan lima transaksi yang terdiri atas dua transaksi Laptop, dua Monitor, dan satu Printer. Kondisi ini membandingkan harga satuan produk, sehingga jumlah barang yang dibeli belum memengaruhi hasil penyaringan.

Kolom `total` ditambahkan melalui `withColumn()` dengan mengalikan harga dan jumlah barang pada setiap baris. Sebagai contoh, transaksi Laptop dengan ID 1 menghasilkan Rp15.000.000 dari harga Rp7.500.000 dikalikan 2 unit. Setelah data diurutkan menggunakan `orderBy()` secara menurun, transaksi tersebut berada di posisi pertama. Berikutnya terdapat Laptop ID 8 dan Monitor ID 10 dengan nilai masing-masing Rp7.500.000. Transaksi dengan nilai terkecil adalah Mouse ID 2, yaitu Rp750.000.

## E.3 Agregasi Data Menggunakan DataFrame API

Total pendapatan, jumlah transaksi, dan rata-rata nilai transaksi dihitung per kategori, lalu statistik untuk seluruh data.

**Langkah 1.** Total pendapatan setiap kategori.

In [ ]:
from pyspark.sql.functions import sum

df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan")
    ).show()

+----------+----------------+
|  kategori|total_pendapatan|
+----------+----------------+
|Elektronik|        36800000|
| Aksesoris|         4650000|
+----------+----------------+



**Langkah 2.** Jumlah transaksi setiap kategori.

In [ ]:
from pyspark.sql.functions import count

df_total.groupBy("kategori") \
    .agg(
        count("*").alias("jumlah_transaksi")
    ).show()

+----------+----------------+
|  kategori|jumlah_transaksi|
+----------+----------------+
|Elektronik|               5|
| Aksesoris|               5|
+----------+----------------+



**Langkah 3.** Rata-rata nilai transaksi.

In [ ]:
from pyspark.sql.functions import avg

df_total.groupBy("kategori") \
    .agg(
        avg("total").alias("rata_rata_transaksi")
    ).show()

+----------+-------------------+
|  kategori|rata_rata_transaksi|
+----------+-------------------+
|Elektronik|          7360000.0|
| Aksesoris|           930000.0|
+----------+-------------------+



**Langkah 4.** Statistik keseluruhan.

In [ ]:
df_total.select(
    sum("total").alias("total_penjualan"),
    avg("total").alias("rata_rata_penjualan"),
    count("*").alias("jumlah_transaksi")
).show()

+---------------+-------------------+----------------+
|total_penjualan|rata_rata_penjualan|jumlah_transaksi|
+---------------+-------------------+----------------+
|       41450000|          4145000.0|              10|
+---------------+-------------------+----------------+



### Pembahasan E.3

Pengelompokan berdasarkan kategori menghasilkan total pendapatan Elektronik sebesar Rp36.800.000 dan Aksesoris sebesar Rp4.650.000. Masing-masing kategori memiliki lima transaksi, dengan rata-rata nilai transaksi Rp7.360.000 untuk Elektronik dan Rp930.000 untuk Aksesoris. Secara keseluruhan, 10 transaksi menghasilkan penjualan Rp41.450.000 dengan rata-rata Rp4.145.000 per transaksi.

#### Pertanyaan analisis

1. Elektronik merupakan kategori dengan pendapatan terbesar. Kedua kategori memiliki jumlah transaksi yang sama, tetapi nilai setiap transaksi Elektronik lebih tinggi pada dataset ini. Nilai tersebut diperoleh dari perkalian harga satuan dengan jumlah barang. Harga Laptop, Monitor, dan Printer yang lebih tinggi menjadi faktor utama besarnya pendapatan Elektronik.
2. Total pendapatan seluruh transaksi adalah Rp41.450.000, yang diperoleh dari penjumlahan pendapatan Elektronik sebesar Rp36.800.000 dan Aksesoris sebesar Rp4.650.000.
3. `groupBy("kategori")` mengelompokkan baris berdasarkan kategori sebelum agregasi dihitung. Dengan cara ini, pendapatan dan rata-rata transaksi setiap kategori dapat dibandingkan. Jika agregasi dilakukan langsung pada seluruh DataFrame tanpa pengelompokan, hasilnya berupa ringkasan keseluruhan data.
4. Fungsi `sum("total")` menjumlahkan nilai transaksi, `count("*")` menghitung banyaknya baris transaksi, dan `avg("total")` menghitung rata-rata nilai transaksi. Pada kategori Elektronik, ketiganya menghasilkan Rp36.800.000, 5 transaksi, dan Rp7.360.000. Jumlah transaksi ini berbeda dari jumlah unit barang, yang tersimpan pada kolom `jumlah`.

## E.4 Pemrosesan Data Menggunakan Spark SQL

DataFrame `df_total` didaftarkan sebagai temporary view bernama `transaksi`, lalu diolah dengan query SQL.

**Langkah 1.** Membuat temporary view.

In [ ]:
df_total.createOrReplaceTempView(
    "transaksi"
)

**Langkah 2.** Menampilkan seluruh data.

In [ ]:
spark.sql("""
    SELECT *
    FROM transaksi
""").show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



**Langkah 3.** Filtering menggunakan SQL.

In [ ]:
spark.sql("""
    SELECT produk, kategori, harga, jumlah, total
    FROM transaksi
    WHERE total > 1000000
""").show()

+-------+----------+-------+------+--------+
| produk|  kategori|  harga|jumlah|   total|
+-------+----------+-------+------+--------+
| Laptop|Elektronik|7500000|     2|15000000|
|Monitor|Elektronik|2500000|     2| 5000000|
|Printer|Elektronik|1800000|     1| 1800000|
| Laptop|Elektronik|7500000|     1| 7500000|
|  Mouse| Aksesoris| 150000|     8| 1200000|
|Monitor|Elektronik|2500000|     3| 7500000|
+-------+----------+-------+------+--------+



**Langkah 4.** Agregasi menggunakan SQL.

In [ ]:
spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""").show()

+----------+----------------+----------------+-------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_transaksi|
+----------+----------------+----------------+-------------------+
|Elektronik|        36800000|               5|          7360000.0|
| Aksesoris|         4650000|               5|           930000.0|
+----------+----------------+----------------+-------------------+



**Langkah 5.** Mengurutkan hasil query.

In [ ]:
spark.sql("""
    SELECT produk,
           SUM(total) AS total_penjualan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_penjualan DESC
""").show()

+---------+---------------+
|   produk|total_penjualan|
+---------+---------------+
|   Laptop|       22500000|
|  Monitor|       12500000|
|    Mouse|        1950000|
|  Printer|        1800000|
|  Headset|        1000000|
| Keyboard|         900000|
|Flashdisk|         800000|
+---------+---------------+



### Pembahasan E.4

DataFrame `df_total` didaftarkan sebagai temporary view bernama `transaksi` agar dapat diakses melalui Spark SQL. Query `SELECT * FROM transaksi` menampilkan data yang sama dengan `df_total`, termasuk kolom `total` yang telah dihitung sebelumnya.

Penyaringan dengan `WHERE total > 1000000` menghasilkan enam transaksi. Mouse ID 9 masuk karena total pembeliannya Rp1.200.000, meskipun harga satuannya Rp150.000. Headset tidak masuk karena totalnya tepat Rp1.000.000, sedangkan kondisi query menggunakan operator lebih besar (`>`). Hasil ini berbeda dari penyaringan pada E.2 yang menghasilkan lima transaksi karena kondisi di sana menggunakan kolom `harga`.

Setelah transaksi dikelompokkan berdasarkan produk dan total penjualannya diurutkan secara menurun, Laptop menempati urutan pertama dengan Rp22.500.000. Nilai tersebut berasal dari dua transaksi sebesar Rp15.000.000 dan Rp7.500.000. Monitor berada di urutan kedua dengan Rp12.500.000, disusul Mouse sebesar Rp1.950.000.

#### Pertanyaan

Hasil agregasi Spark SQL sesuai dengan DataFrame API pada E.3. Kategori Elektronik menghasilkan Rp36.800.000 dari lima transaksi dengan rata-rata Rp7.360.000, sedangkan Aksesoris menghasilkan Rp4.650.000 dari lima transaksi dengan rata-rata Rp930.000. Kesamaan hasil ini sesuai dengan data, pengelompokan, dan fungsi agregasi yang digunakan pada kedua pendekatan.

Spark SQL dan DataFrame API sama-sama diproses melalui mesin Spark SQL dan optimizer Catalyst. Pengguna dapat memilih cara penulisan sesuai kebutuhan. SQL memudahkan penulisan analisis berbentuk query, sementara DataFrame API dapat dirangkai langsung dengan operasi lain dalam kode Python.

## F. Eksperimen Lazy Evaluation

Sebuah filter didefinisikan dulu tanpa action. Setelah itu `show()` dipanggil, lalu rencana eksekusinya dilihat dengan `explain()`.

In [ ]:
df_filter = df.filter(
    df.harga > 1000000
)

print("Transformation selesai")

Transformation selesai


In [ ]:
df_filter.show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



In [ ]:
df_filter.explain()

== Physical Plan ==
*(1) Filter (isnotnull(harga#3L) AND (harga#3L > 1000000))
+- *(1) Scan ExistingRDD[id_transaksi#0L,produk#1,kategori#2,harga#3L,jumlah#4L]




### Pembahasan F

Pada sel pertama, perintah `df.filter(df.harga > 1000000)` membentuk DataFrame `df_filter`, kemudian program mencetak "Transformation selesai". Pesan tersebut berasal dari `print()` dan tidak menunjukkan bahwa penyaringan data sudah dieksekusi. Pemrosesan untuk menampilkan hasil baru dipicu ketika `df_filter.show()` dipanggil. Outputnya berisi lima transaksi dengan harga satuan di atas Rp1.000.000.

Hasil `explain()` memperlihatkan physical plan berupa `Scan ExistingRDD` yang diikuti `Filter`. Kondisi filternya adalah `isnotnull(harga)` dan `harga > 1000000`, sehingga baris dengan harga kosong maupun harga yang tidak memenuhi batas dikeluarkan. Perintah `explain()` menampilkan rencana eksekusi tanpa mengumpulkan hasil transaksi seperti `show()`.

1. Lazy evaluation memungkinkan Spark menyusun rangkaian operasi sebelum memproses data. Pada DataFrame, optimizer dapat meninjau rencana tersebut dan menerapkan optimasi yang sesuai sebelum action dijalankan.
2. Pada pengolahan data besar, optimasi dapat mencakup pemilihan kolom yang diperlukan, penempatan filter lebih awal, serta penggabungan operasi yang kompatibel. Manfaatnya bergantung pada query dan sumber data yang digunakan. Eksperimen ini memperlihatkan penundaan eksekusi dan physical plan, tetapi belum mengukur penghematan waktu atau transfer data pada cluster.
3. Execution plan membantu menjelaskan urutan pemrosesan yang direncanakan Spark. Melalui `explain()`, pengguna dapat memeriksa operasi pembacaan, penyaringan, dan operasi lain sebelum menilai bagian yang perlu dioptimalkan.
4. Pendefinisian transformation tetap memerlukan waktu untuk membentuk objek dan rencana, tetapi belum menjalankan pemrosesan data yang ditunda. Action kemudian memicu pekerjaan sesuai hasil yang dibutuhkan. `show()` mengambil data untuk ditampilkan, sedangkan `count()` menghitung jumlah seluruh baris. Karena percobaan ini tidak mencatat durasi, perbedaan waktu keduanya belum dapat dinyatakan secara kuantitatif.

## Tugas Praktikum 6

Tugas memakai dataset transaksi dari bagian E.2, yaitu `df`, `df_total`, dan temporary view `transaksi`. Tugas 1 sampai 4 dikerjakan dengan kode, sedangkan Tugas 5 berupa jawaban konsep.

### Tugas 1. Analisis Produk

Tampilkan lima produk dengan total pendapatan tertinggi menggunakan DataFrame API.

In [ ]:
df_total.groupBy("produk") \
    .agg(
        sum("total").alias("total_pendapatan")
    ) \
    .orderBy(col("total_pendapatan").desc()) \
    .limit(5) \
    .show()

+-------+----------------+
| produk|total_pendapatan|
+-------+----------------+
| Laptop|        22500000|
|Monitor|        12500000|
|  Mouse|         1950000|
|Printer|         1800000|
|Headset|         1000000|
+-------+----------------+



### Pembahasan Tugas 1

Produk dikelompokkan dengan `groupBy()`, total pendapatannya dijumlahkan dengan `sum()`, lalu hasilnya diurutkan menurun dan dibatasi lima baris dengan `limit(5)`. Laptop berada di urutan pertama dengan Rp22.500.000, disusul Monitor Rp12.500.000, Mouse Rp1.950.000, Printer Rp1.800.000, dan Headset Rp1.000.000. Dataset hanya memuat tujuh produk, sehingga Keyboard (Rp900.000) dan Flashdisk (Rp800.000) tidak masuk lima besar.

### Tugas 2. Analisis Kategori

Hitung total pendapatan, jumlah transaksi, dan rata-rata jumlah produk yang terjual pada setiap kategori.

In [ ]:
df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan"),
        count("*").alias("jumlah_transaksi"),
        avg("jumlah").alias("rata_rata_jumlah")
    ).show()

+----------+----------------+----------------+----------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_jumlah|
+----------+----------------+----------------+----------------+
|Elektronik|        36800000|               5|             1.8|
| Aksesoris|         4650000|               5|             6.0|
+----------+----------------+----------------+----------------+



### Pembahasan Tugas 2

Tiga ukuran dihitung sekaligus dalam satu `agg()`. Elektronik menghasilkan Rp36.800.000 dari 5 transaksi dengan rata-rata 1,8 unit per transaksi. Aksesoris menghasilkan Rp4.650.000 dari 5 transaksi dengan rata-rata 6,0 unit per transaksi. Aksesoris dibeli dalam jumlah lebih banyak, tetapi nilainya jauh lebih kecil karena harga satuannya rendah.

### Tugas 3. Filtering Data

Tampilkan transaksi dengan harga lebih dari Rp100.000 dan jumlah pembelian minimal 3 unit.

In [ ]:
df_total.filter(
    (col("harga") > 100000) & (col("jumlah") >= 3)
).show()

+------------+--------+----------+-------+------+-------+
|id_transaksi|  produk|  kategori|  harga|jumlah|  total|
+------------+--------+----------+-------+------+-------+
|           2|   Mouse| Aksesoris| 150000|     5| 750000|
|           3|Keyboard| Aksesoris| 300000|     3| 900000|
|           7| Headset| Aksesoris| 250000|     4|1000000|
|           9|   Mouse| Aksesoris| 150000|     8|1200000|
|          10| Monitor|Elektronik|2500000|     3|7500000|
+------------+--------+----------+-------+------+-------+



### Pembahasan Tugas 3

Kedua syarat digabung dengan operator `&`, dan tiap kondisi diberi tanda kurung karena prioritas operator pada Python. Hasilnya lima transaksi, yaitu ID 2, 3, 7, 9, dan 10. Laptop dan Printer tidak masuk karena dibeli kurang dari 3 unit walaupun harganya tinggi, dan Monitor ID 4 tidak masuk karena hanya 2 unit. Flashdisk tidak masuk karena harganya Rp80.000 sehingga tidak lebih dari Rp100.000, walaupun dibeli 10 unit.

### Tugas 4. Spark SQL

Buat query SQL untuk menampilkan nama produk dan total pendapatan setiap produk, diurutkan dari pendapatan terbesar.

In [ ]:
spark.sql("""
    SELECT produk,
           SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

+---------+----------------+
|   produk|total_pendapatan|
+---------+----------------+
|   Laptop|        22500000|
|  Monitor|        12500000|
|    Mouse|         1950000|
|  Printer|         1800000|
|  Headset|         1000000|
| Keyboard|          900000|
|Flashdisk|          800000|
+---------+----------------+



### Pembahasan Tugas 4

Query dijalankan pada temporary view `transaksi` dan menghasilkan tujuh produk. Lima baris teratas sama dengan hasil Tugas 1 yang memakai DataFrame API, sedangkan Keyboard Rp900.000 dan Flashdisk Rp800.000 muncul pada dua urutan terakhir. Kecocokan ini menunjukkan bahwa SQL dan DataFrame API memberi hasil yang sama untuk agregasi per produk.

### Tugas 5. Analisis Konsep

Jelaskan perbedaan RDD dan DataFrame serta berikan contoh kondisi penggunaan masing-masing.

**Jawaban.** RDD adalah abstraksi dasar Spark berupa kumpulan elemen terdistribusi. Operasinya ditulis dengan fungsi seperti `map()` dan `filter()`, dan Spark memperlakukan setiap elemen apa adanya tanpa mengetahui struktur kolomnya. DataFrame disusun dalam baris dan kolom serta memiliki skema, sehingga Spark mengetahui nama dan tipe tiap kolom. API-nya, seperti `select()`, `filter()`, `groupBy()`, dan `agg()`, lebih mudah dipakai untuk data terstruktur.

RDD cocok dipakai ketika data tidak berbentuk tabel atau ketika setiap elemen perlu diproses langsung dengan fungsi sendiri, misalnya list angka pada bagian E.1 atau baris teks mentah. DataFrame cocok untuk data bertabel seperti transaksi penjualan pada praktikum ini. Total per kategori cukup ditulis dengan `groupBy()` dan `agg()`, atau dengan query SQL, tanpa mengatur sendiri pasangan key dan value.

## Pertanyaan Evaluasi

**1. Apa yang dimaksud dengan Apache Spark dan apa keunggulannya?** Apache Spark adalah framework komputasi terdistribusi untuk memproses data dalam jumlah besar. Keunggulannya antara lain pemrosesan memakai memori untuk mempercepat komputasi, dukungan pemrosesan paralel dan terdistribusi, dukungan bahasa Python, Scala, Java, dan R, serta API tingkat tinggi seperti DataFrame dan SQL.

**2. Perbedaan transformation dan action pada RDD.** Transformation menghasilkan RDD baru dan belum dijalankan sampai ada action. Contohnya `map()` untuk mengalikan setiap elemen dengan 2 dan `filter()` untuk mengambil elemen lebih besar dari 50. Action memicu eksekusi dan mengembalikan hasil ke driver. Contohnya `count()` yang menghasilkan 10 elemen dan `collect()` yang mengembalikan seluruh isi RDD.

**3. Mengapa Spark menerapkan lazy evaluation?** Spark menunda eksekusi supaya seluruh rangkaian operasi dapat disusun menjadi satu rencana sebelum data diproses. Pada DataFrame, rencana ini dapat dioptimalkan lebih dulu. Pada bagian F, pesan "Transformation selesai" tercetak sebelum data tampil, sedangkan `explain()` menunjukkan rencana tanpa menjalankan `show()`.

**4. Perbedaan RDD dan DataFrame.** RDD adalah kumpulan elemen terdistribusi tanpa skema dan dioperasikan dengan fungsi per elemen, misalnya `lambda x: x > 50`. DataFrame berbentuk baris dan kolom dengan skema yang dapat dilihat lewat `printSchema()`, dan dioperasikan dengan nama kolom, misalnya `df.harga > 1000000`.

**5. Mengapa DataFrame lebih mudah untuk data terstruktur?** Data terstruktur sudah memiliki kolom dan tipe, sehingga DataFrame menyediakan operasi tingkat tinggi seperti `select()`, `filter()`, `groupBy()`, `agg()`, `orderBy()`, dan `join()`. Total pendapatan per kategori cukup dihitung dengan satu rangkaian `groupBy()` dan `agg()`. Dengan RDD, pengelompokan dan penjumlahan harus ditulis sendiri melalui fungsi pada setiap elemen.

**6. Hubungan Spark SQL dan DataFrame API.** Keduanya bekerja pada data terstruktur yang sama. `df_total` didaftarkan sebagai temporary view, lalu diquery dengan `spark.sql()`, dan hasil query kembali berupa DataFrame. Hasil agregasi per kategori pada E.3 dan E.4 sama persis, dan keduanya diproses oleh mesin Spark SQL dengan optimizer Catalyst.

**7. Bagaimana Spark mempercepat pemrosesan dibandingkan pendekatan berbasis disk?** Spark memakai memori untuk menyimpan data selama pemrosesan, sehingga data tidak harus dibaca dan ditulis ke disk pada setiap tahap, dan pekerjaan dibagi secara paralel. Percobaan ini memakai 10 baris data pada satu mesin dan tidak mengukur waktu, jadi percepatan tersebut hanya dijelaskan secara konsep dan tidak dibuktikan oleh angka dari praktikum.